# Mamba — TaskTracker-only

Pipeline dung 470 TaskTracker sessions va ba split CSV co dinh cua nhom. Test bat buoc 69 sessions (15 label 1, 54 label 0). Notebook khong tao split moi. PasteTrace khong duoc su dung.

In [ ]:
# Cell 1: clone/pull dung source tu GitHub
from pathlib import Path
import os, subprocess

REPO_URL = 'https://github.com/lequocviet-3103/Fraud-Detection.git'
BRANCH = 'ModelMambaV2'
REPO_DIR = Path('/kaggle/working/Fraud-Detection')

if (REPO_DIR / '.git').is_dir():
    subprocess.run(['git', '-C', str(REPO_DIR), 'fetch', 'origin', BRANCH], check=True)
    subprocess.run(['git', '-C', str(REPO_DIR), 'checkout', BRANCH], check=True)
    subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only', 'origin', BRANCH], check=True)
elif REPO_DIR.exists() and any(REPO_DIR.iterdir()):
    raise RuntimeError(f'{REPO_DIR} exists but is not a git repository; remove/rename it first.')
else:
    subprocess.run(['git', 'clone', '--branch', BRANCH, '--single-branch', REPO_URL, str(REPO_DIR)], check=True)

os.chdir(REPO_DIR)
commit = subprocess.check_output(['git', 'rev-parse', '--short', 'HEAD'], text=True).strip()
print('Working directory:', Path.cwd())
print('Branch:', BRANCH, '| commit:', commit)

In [ ]:
!pip install -q -r requirements-mamba.txt
import torch
from mamba_ssm import Mamba
print('CUDA:', torch.cuda.is_available(), '| torch:', torch.__version__)
if not torch.cuda.is_available():
    raise RuntimeError('CUDA is not enabled. In Kaggle: Settings -> Accelerator -> GPU.')
device = torch.device('cuda')
probe_model = Mamba(d_model=64).to(device)
probe_input = torch.randn(2, 32, 64, device=device)
with torch.no_grad():
    probe_output = probe_model(probe_input)
print('Mamba output:', probe_output.shape, '| device:', probe_output.device)
del probe_model, probe_input, probe_output
torch.cuda.empty_cache()

In [ ]:
!python -m src.data.build_sequences --min-events 1
!python -m src.data.make_splits  # validate only; never creates a split

In [ ]:
import pandas as pd
for name in ['train', 'validation', 'test']:
    frame = pd.read_csv(f'data/splits/tasktracker/{name}.csv')
    print(name, len(frame))

## Train + validation
Scaler chi fit train. Checkpoint va threshold duoc chon bang validation. Test split khong duoc nap trong qua trinh nay.

In [ ]:
!python -m src.models.mamba_model train \
    --d-model 64 --n-layers 2 --dropout 0.2 \
    --epochs 80 --lr 3e-4 --patience 10 \
    --batch-size 8 --max-len 1000 --seed 42

## Held-out TaskTracker test
Lenh nay tao `results/mamba/predictions.csv` va `results/mamba/metrics.json`.

In [ ]:
!python -m src.models.mamba_model test
import json, pandas as pd
display(pd.read_csv('results/mamba/predictions.csv').head())
with open('results/mamba/metrics.json', encoding='utf8') as f:
    metrics = json.load(f)
metrics